# Structured generation: step by step

In [1]:
from outlines import models, generate
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from enum import Enum
import pandas as pd
import json
from openai import OpenAI


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/thomaspesneau/Documents/Projects/llm_classifier_outlines/.venv/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/thomaspesneau/Documents/Projects/llm_classifier_outlines/.venv/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/thomaspesneau/Documents/Projects/llm_cl

In [2]:
load_dotenv()

True

## Define prompt

In [3]:
country_codes = pd.read_csv("../country_codes_mapping.csv").loc[:, ["alpha-3"]].to_numpy().ravel()
CountriesEnum: Enum = Enum("Countries", {code: code for code in country_codes})

In [4]:
class OlympicMedals(BaseModel):
    country: CountriesEnum
    year: int
    location: str
    gold: int
    silver: int
    bronze: int

In [5]:
prompt = f"""
You are a helpful assistant who outputs a valid JSON from a given text about olympic medals.    
You MUST follow the given JSON schema: 
{OlympicMedals.model_json_schema()}

Example:
User: "France has won 16 gold medals, 26 silver medals, and 22 bronze medals during the 2024 Paris Olympic games"
Result: {OlympicMedals(country=CountriesEnum.FRA, year=2024, location="Paris", gold=16, silver=26, bronze=22).model_dump_json()}
"""

In [6]:
print(prompt)


You are a helpful assistant who outputs a valid JSON from a given text about olympic medals.    
You MUST follow the given JSON schema: 
{'$defs': {'Countries': {'enum': ['AFG', 'ALA', 'ALB', 'DZA', 'ASM', 'AND', 'AGO', 'AIA', 'ATA', 'ATG', 'ARG', 'ARM', 'ABW', 'AUS', 'AUT', 'AZE', 'BHS', 'BHR', 'BGD', 'BRB', 'BLR', 'BEL', 'BLZ', 'BEN', 'BMU', 'BTN', 'BOL', 'BES', 'BIH', 'BWA', 'BVT', 'BRA', 'IOT', 'BRN', 'BGR', 'BFA', 'BDI', 'CPV', 'KHM', 'CMR', 'CAN', 'CYM', 'CAF', 'TCD', 'CHL', 'CHN', 'CXR', 'CCK', 'COL', 'COM', 'COG', 'COD', 'COK', 'CRI', 'CIV', 'HRV', 'CUB', 'CUW', 'CYP', 'CZE', 'DNK', 'DJI', 'DMA', 'DOM', 'ECU', 'EGY', 'SLV', 'GNQ', 'ERI', 'EST', 'SWZ', 'ETH', 'FLK', 'FRO', 'FJI', 'FIN', 'FRA', 'GUF', 'PYF', 'ATF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GIB', 'GRC', 'GRL', 'GRD', 'GLP', 'GUM', 'GTM', 'GGY', 'GIN', 'GNB', 'GUY', 'HTI', 'HMD', 'VAT', 'HND', 'HKG', 'HUN', 'ISL', 'IND', 'IDN', 'IRN', 'IRQ', 'IRL', 'IMN', 'ISR', 'ITA', 'JAM', 'JPN', 'JEY', 'JOR', 'KAZ', 'KEN', 'KIR', 'P

## Generate JSON from text

### Without outlines

In [7]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_json_response_without_outlines(user_request: str, client: OpenAI = client):
    completion = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "developer", "content": prompt},
            {
                "role": "user",
                "content": user_request,
            },
        ],
    )

    return completion.choices[0].message.content

### With outlines

In [8]:
model = models.openai(
    "gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY")
)
generator = generate.json(model, OlympicMedals)

def get_json_response_with_outlines(user_request: str, generator: generate.api.SequenceGeneratorAdapter = generator):
    response: OlympicMedals = generator(user_request)
    return response.model_dump_json()

## Evaluation

In [14]:
def is_valid_json(response: str):
    try:
        json.loads(response)
        return 1
    except ValueError:
        return 0

In [15]:
resp1 = get_json_response_without_outlines("During the 2020 Tokyo Olympics, which actually took place in 2021 due to COVID pandemic, Canada impressed with a record 12 gold medals for its athletes, when previously it had only won 4. In addition to that, they got 23 silver, and 15 bronze medals")

In [16]:
resp2 = get_json_response_with_outlines("During the 2020 Tokyo Olympics, which actually took place in 2021 due to COVID pandemic, Canada impressed with a record 12 gold medals for its athletes, when previously it had only won 4. In addition to that, they got 23 silver, and 15 bronze medals")